# CISEI Scenario Planner

Notebook completo para:

1. Configurar o projeto.
2. Inicializar o `ScenarioAgentOpenAI` com o contexto do plugin.
3. Conversar com o agente para construir o cenário.
4. Validar e confirmar o cenário.
5. Gerar `scenario.toml`.
6. Carregar `nodes.csv`.
7. Visualizar os nós no mapa.
8. Executar o planejamento através do `PlannerOrchestrator`.
9. Obter o `planning_result` final.
10. Sobrepor os enlaces planejados no mapa.

## Arquitetura

```text
Usuário
   │
   ▼
ScenarioAgentOpenAI
   │
   │ interpreta intenção
   ▼
ScenarioAgent
   │
   ▼
ScenarioConversation
   │
   ▼
ScenarioSession
   │
   ▼
ScenarioBuilder
   │
   ▼
scenario.toml
   │
   │
   └──────────────────────────────┐
                                  │
                                  ▼
                         PlannerOrchestrator
                                  │
                                  ▼
                                MCP
                                  │
             ┌────────────────────┼────────────────────┐
             ▼                    ▼                    ▼
      planning_open        planning_run        planning_status
                                                       │
                                                       ▼
                                               planning_result
                                                       │
                                                       ▼
                                               planning_export

In [ ]:
## Célula 2 — Código — configuração do projeto
from pathlib import Path

# ============================================================
# CONFIGURAÇÃO DO PROJETO
# ============================================================

PROJECT_NAME = "curitiba_lte_2"

WORKSPACE_ROOT = Path("/home/jovyan/work/cisei_workspace")
PROJECTS_ROOT = WORKSPACE_ROOT / "projects"

PROJECT_ROOT = PROJECTS_ROOT / PROJECT_NAME

SCENARIO_PATH = PROJECT_ROOT / "scenario.toml"
NODES_PATH = PROJECT_ROOT / "nodes.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto configurado")
print()
print(f"PROJECT_NAME : {PROJECT_NAME}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SCENARIO_PATH: {SCENARIO_PATH}")
print(f"NODES_PATH   : {NODES_PATH}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")
print(f"REPORTS_DIR  : {REPORTS_DIR}")

In [ ]:
import sys

# ============================================================
# IMPORTS DO PLANAPP
# ============================================================

PROJECT_CODE_ROOT = Path("/home/jovyan/work/planapp-mcp")
PLUGIN_ROOT = PROJECT_CODE_ROOT / "cisei-planning-engineering"

if str(PROJECT_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE_ROOT))

from scenario_agent_openai import create_scenario_agent_openai
from scenario_domain.providers import CISEIPluginDomainContextProvider


# ============================================================
# CONTEXTO DO PLUGIN
# ============================================================

DOMAIN_CONTEXT_SECTIONS = (
    "scenario_format",
    "assistant_workflow",
    "design_gaps_and_decisions",
)

provider = CISEIPluginDomainContextProvider(PLUGIN_ROOT)


# ============================================================
# SCENARIO AGENT OPENAI
# ============================================================

agent = create_scenario_agent_openai(
    PROJECT_ROOT,
    domain_context_provider=provider,
    domain_context_sections=DOMAIN_CONTEXT_SECTIONS,
)

print("ScenarioAgentOpenAI inicializado")
print()
print(f"Projeto : {PROJECT_ROOT}")
print(f"Plugin  : {PLUGIN_ROOT}")
print(f"Seções  : {DOMAIN_CONTEXT_SECTIONS}")

## Verificação do contexto do plugin

Esta célula é apenas diagnóstica.

O plugin fornece contexto de domínio para o agente, mas não executa o planejamento e não substitui o MCP.

In [ ]:
# ============================================================
# VERIFICAÇÃO DO CONTEXTO DO PLUGIN
# ============================================================

try:
    print("Plugin root:")
    print(PLUGIN_ROOT)
    print()

    print("Seções solicitadas:")
    for section in DOMAIN_CONTEXT_SECTIONS:
        print(f"  - {section}")

    print()
    print("Provider:")
    print(type(provider).__name__)

except Exception as exc:
    print("Erro ao verificar o contexto do plugin:")
    print(repr(exc))

# Conversa com o ScenarioAgentOpenAI

Nesta etapa o agente trabalha exclusivamente na construção do cenário.

A conversa é síncrona.

O planejamento MCP ainda não é executado nesta etapa.

### Exemplos

Você pode começar com:

> Quero criar um cenário LTE para Curitiba usando o arquivo nodes.csv.

Depois, fornecer os parâmetros desejados.

O agente deve solicitar os valores que estiverem faltando em vez de inventar parâmetros técnicos.

In [ ]:
import html
import json
import ipywidgets as widgets

from IPython.display import display


# ============================================================
# ESTADO DA CONVERSA
# ============================================================

def obter_estado():
    return agent.scenario_agent.conversation_state()


def renderizar_estado():
    estado = obter_estado()

    print(json.dumps(
        estado,
        indent=2,
        ensure_ascii=False,
        default=str,
    ))


# ============================================================
# CONVERSA
# ============================================================

def conversar(mensagem: str):
    mensagem = str(mensagem).strip()

    if not mensagem:
        return None

    resposta = agent.chat(mensagem)

    return resposta


# ============================================================
# WIDGETS
# ============================================================

entrada = widgets.Textarea(
    placeholder="Digite sua mensagem para o ScenarioAgentOpenAI...",
    layout=widgets.Layout(
        width="100%",
        min_height="100px",
    ),
)

botao_enviar = widgets.Button(
    description="Enviar",
    button_style="primary",
)

botao_limpar = widgets.Button(
    description="Limpar saída",
)

conversation_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc",
        padding="10px",
        max_height="500px",
        overflow="auto",
    )
)

state_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc",
        padding="10px",
        max_height="400px",
        overflow="auto",
    )
)


# ============================================================
# CALLBACK
# ============================================================

def enviar_mensagem(_):
    mensagem = entrada.value.strip()

    if not mensagem:
        return

    with conversation_output:
        print("=" * 80)
        print("USUÁRIO")
        print("=" * 80)
        print(mensagem)
        print()

        try:
            resposta = conversar(mensagem)

            print("=" * 80)
            print("SCENARIO AGENT")
            print("=" * 80)

            if isinstance(resposta, (dict, list)):
                print(json.dumps(
                    resposta,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                ))
            else:
                print(resposta)

        except Exception as exc:
            print("ERRO:")
            print(repr(exc))

        print()

    entrada.value = ""

    with state_output:
        state_output.clear_output(wait=True)

        try:
            renderizar_estado()
        except Exception as exc:
            print("Erro ao obter estado:")
            print(repr(exc))


def limpar_conversa(_):
    conversation_output.clear_output()
    state_output.clear_output()


botao_enviar.on_click(enviar_mensagem)
botao_limpar.on_click(limpar_conversa)


# ============================================================
# DISPLAY
# ============================================================

display(
    widgets.VBox([
        entrada,
        widgets.HBox([
            botao_enviar,
            botao_limpar,
        ]),
        widgets.HTML("<h3>Conversa</h3>"),
        conversation_output,
        widgets.HTML("<h3>Estado atual do cenário</h3>"),
        state_output,
    ])
)

## Roteiro sugerido para a conversa

Para testar o fluxo atual, podemos usar uma conversa como:

### Mensagem 1

Quero criar um cenário LTE para Curitiba usando o arquivo nodes.csv.

### Mensagem 2

Forneça os parâmetros solicitados pelo agente.

### Mensagem 3

Quando o cenário estiver completo:

Mostre o estado atual do cenário.
### Mensagem 4

Quando tudo estiver correto:

Confirmo o cenário.

Importante: os valores técnicos devem vir do usuário/conversa.
O agente não deve transformar os exemplos do plugin em defaults técnicos.

In [ ]:
## Célula 9 — Código — inspeção do estado
# ============================================================
# INSPECIONAR ESTADO ATUAL
# ============================================================

estado_atual = obter_estado()

print(json.dumps(
    estado_atual,
    indent=2,
    ensure_ascii=False,
    default=str,
))

In [ ]:
# ============================================================
# VALIDAÇÃO ANTES DA CONFIRMAÇÃO
# ============================================================

def verificar_pronto_para_confirmacao():
    estado = obter_estado()

    print(json.dumps(
        estado,
        indent=2,
        ensure_ascii=False,
        default=str,
    ))

    print()
    print("=" * 80)

    valid = estado.get("valid")
    complete = estado.get("complete")
    confirmed = estado.get("confirmed")

    print(f"valid     : {valid}")
    print(f"complete  : {complete}")
    print(f"confirmed : {confirmed}")

    if valid and complete:
        print()
        print("OK: o cenário está válido e completo.")
        print("Pode ser confirmado.")
    else:
        print()
        print("ATENÇÃO: o cenário ainda não está pronto para confirmação.")

    print("=" * 80)


verificar_pronto_para_confirmacao()

In [ ]:
# ============================================================
# CONFIRMAR CENÁRIO
# ============================================================

def confirmar_cenario():
    estado = obter_estado()

    if not estado.get("valid"):
        raise RuntimeError(
            "O cenário não está válido. Corrija os problemas antes da confirmação."
        )

    if not estado.get("complete"):
        raise RuntimeError(
            "O cenário ainda está incompleto. "
            "Preencha os parâmetros faltantes antes da confirmação."
        )

    resposta = conversar("Confirmo o cenário.")

    print(json.dumps(
        resposta,
        indent=2,
        ensure_ascii=False,
        default=str,
    ))

    print()
    print("Estado após confirmação:")
    renderizar_estado()


confirmar_cenario()

In [ ]:
# ============================================================
# AUDITORIA DA CONVERSA
# ============================================================

def mostrar_mensagens():
    estado = obter_estado()

    mensagens = estado.get("messages", [])

    print(f"Total de mensagens: {len(mensagens)}")
    print()

    for i, mensagem in enumerate(mensagens, start=1):
        print("=" * 80)
        print(f"MENSAGEM {i}")
        print("=" * 80)

        if isinstance(mensagem, dict):
            print(json.dumps(
                mensagem,
                indent=2,
                ensure_ascii=False,
                default=str,
            ))
        else:
            print(mensagem)

        print()


mostrar_mensagens()

# Geração do scenario.toml

Depois que o cenário estiver:

- válido;
- completo;
- confirmado;

o `ScenarioBuilder` gera o TOML.

O notebook apenas salva o resultado no projeto.

Não serão criados parâmetros técnicos automaticamente nesta etapa.

In [ ]:
# ============================================================
# GERAR E SALVAR scenario.toml
# ============================================================

def gerar_e_salvar_scenario():
    estado = obter_estado()

    if not estado.get("valid"):
        raise RuntimeError(
            "O cenário não está válido."
        )

    if not estado.get("complete"):
        raise RuntimeError(
            "O cenário não está completo."
        )

    if not estado.get("confirmed"):
        raise RuntimeError(
            "O cenário ainda não foi confirmado."
        )

    toml_text = agent.scenario.generate_toml()

    if not isinstance(toml_text, str) or not toml_text.strip():
        raise RuntimeError(
            "ScenarioBuilder não retornou um TOML válido."
        )

    SCENARIO_PATH.write_text(
        toml_text,
        encoding="utf-8",
    )

    print("scenario.toml gerado com sucesso.")
    print()
    print(f"Caminho:")
    print(SCENARIO_PATH)
    print()
    print(f"Tamanho: {SCENARIO_PATH.stat().st_size} bytes")

    return toml_text


scenario_toml = gerar_e_salvar_scenario()

In [ ]:
# ============================================================
# MOSTRAR scenario.toml
# ============================================================

print("=" * 80)
print(SCENARIO_PATH)
print("=" * 80)
print()

print(SCENARIO_PATH.read_text(encoding="utf-8"))

# Carregamento dos nós

Agora saímos da etapa de construção do cenário.

O arquivo `nodes.csv` é carregado do projeto.

O notebook não cria um `nodes.csv` artificial.

Ele deve existir em:

```text
projects/<scenario_id>/nodes.csv

In [ ]:
## Célula 17 — Código — carregar `nodes.csv`
# ============================================================
# CARREGAR nodes.csv
# ============================================================

import pandas as pd


REQUIRED_NODE_COLUMNS = [
    "site_id",
    "device_profile",
    "kind",
    "lat",
    "lon",
    "x",
    "y",
    "mount_height_m",
    "connected",
    "can_route",
    "rank",
]


def carregar_nodes():
    if not NODES_PATH.exists():
        raise FileNotFoundError(
            f"nodes.csv não encontrado:\n{NODES_PATH}"
        )

    df = pd.read_csv(NODES_PATH)

    missing = [
        col
        for col in REQUIRED_NODE_COLUMNS
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            "Colunas obrigatórias ausentes em nodes.csv:\n"
            + "\n".join(f"- {col}" for col in missing)
        )

    numeric_columns = [
        "lat",
        "lon",
        "x",
        "y",
        "mount_height_m",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    if df[numeric_columns].isna().any().any():
        invalid_columns = [
            col
            for col in numeric_columns
            if df[col].isna().any()
        ]

        raise ValueError(
            "Existem valores inválidos nas colunas numéricas: "
            + ", ".join(invalid_columns)
        )

    return df


nodes = carregar_nodes()

print("nodes.csv carregado com sucesso.")
print()
print(f"Linhas: {len(nodes)}")
print(f"Colunas: {len(nodes.columns)}")
print()
print(nodes.head())

In [ ]:
# ============================================================
# RESUMO DOS NÓS
# ============================================================

print("=" * 80)
print("RESUMO DOS NÓS")
print("=" * 80)

print()
print(f"Total de nós: {len(nodes)}")

print()
print("Tipos:")
print(nodes["kind"].value_counts(dropna=False))

print()
print("Perfis:")
print(nodes["device_profile"].value_counts(dropna=False))

print()
print("Conectados:")
print(nodes["connected"].value_counts(dropna=False))

print()
print("Roteáveis:")
print(nodes["can_route"].value_counts(dropna=False))

# Mapa dos nós

O mapa abaixo mostra os pontos existentes no `nodes.csv`.

Neste momento ainda são exibidos somente os nós.

Os enlaces planejados serão adicionados na segunda metade do notebook, depois que o `planning_result` real estiver disponível.

In [ ]:
# ============================================================
# MAPA DOS NÓS
# ============================================================

from ipyleaflet import Map, CircleMarker, Popup
from ipywidgets import HTML


def criar_mapa_nodes(df):
    if df.empty:
        raise ValueError("nodes.csv está vazio.")

    center_lat = float(df["lat"].mean())
    center_lon = float(df["lon"].mean())

    mapa = Map(
        center=(center_lat, center_lon),
        zoom=13,
        scroll_wheel_zoom=True,
        layout={
            "height": "700px",
            "width": "100%",
        },
    )

    markers = []

    for _, row in df.iterrows():

        if str(row["kind"]).lower() in {
            "tower",
            "torre",
        }:
            marker_color = "red"
        else:
            marker_color = "blue"

        popup_html = f"""
        <b>Site:</b> {html.escape(str(row["site_id"]))}<br>
        <b>Profile:</b> {html.escape(str(row["device_profile"]))}<br>
        <b>Kind:</b> {html.escape(str(row["kind"]))}<br>
        <b>Latitude:</b> {row["lat"]}<br>
        <b>Longitude:</b> {row["lon"]}<br>
        <b>X:</b> {row["x"]}<br>
        <b>Y:</b> {row["y"]}<br>
        <b>Mount height:</b> {row["mount_height_m"]} m<br>
        <b>Connected:</b> {row["connected"]}<br>
        <b>Can route:</b> {row["can_route"]}<br>
        <b>Rank:</b> {row["rank"]}
        """

        popup = Popup(
            child=HTML(value=popup_html),
            close_button=True,
            auto_close=True,
        )

        marker = CircleMarker(
            location=(
                float(row["lat"]),
                float(row["lon"]),
            ),
            radius=7,
            color=marker_color,
            fill_color=marker_color,
            fill_opacity=0.8,
            weight=2,
        )

        marker.popup = popup

        mapa.add(marker)
        markers.append(marker)

    # Ajustar visualização aos pontos
    lat_min = float(df["lat"].min())
    lat_max = float(df["lat"].max())
    lon_min = float(df["lon"].min())
    lon_max = float(df["lon"].max())

    mapa.fit_bounds([
        [lat_min, lon_min],
        [lat_max, lon_max],
    ])

    return mapa


mapa_nodes = criar_mapa_nodes(nodes)

mapa_nodes

# Execução do planejamento

A partir daqui entramos na etapa de planejamento.

O cenário já foi:

- construído pelo ScenarioAgentOpenAI;
- validado;
- confirmado;
- salvo como `scenario.toml`;
- associado ao `nodes.csv`.

O `PlannerOrchestrator` será responsável por executar o planejamento através do MCP.

Fluxo:

```text
scenario.toml
     │
     ▼
PlannerOrchestrator
     │
     ├── planning_open
     │
     ├── planning_run
     │
     └── planning_result
              │
              ▼
       solução final
              │
              ▼
        mapa dos enlaces

In [ ]:
## Célula 22 — Código — inicializar PlannerOrchestrator
# ============================================================
# PLANNER ORCHESTRATOR
# ============================================================

from planner_orchestrator import PlannerOrchestrator


planner_orchestrator = PlannerOrchestrator()

print("PlannerOrchestrator inicializado.")
print()
print(type(planner_orchestrator).__name__)

## Parâmetros de execução

Os parâmetros abaixo são parâmetros de execução do planner.

Eles não são defaults inseridos no `scenario.toml`.

Para este teste:

- `scenario_id`: `curitiba_lte`
- `primary_tech`: `lte`
- `planner`: `cell`
- `rank_threshold`: `10`
- `solution_kind`: `graph`

O `PlannerOrchestrator` pode completar os parâmetros de execução que não forem explicitamente informados.

In [ ]:
# ============================================================
# PREPARAR EXECUÇÃO
# ============================================================

PLANNING_REQUEST = {
    "scenario_id": PROJECT_NAME,
    "primary_tech": "lte",
}

print("Request de planejamento:")
print()

print(json.dumps(
    PLANNING_REQUEST,
    indent=2,
    ensure_ascii=False,
))

# Executar planejamento

Esta é uma célula assíncrona.

Execute-a somente quando:

1. `scenario.toml` estiver correto;
2. `nodes.csv` estiver correto;
3. o cenário tiver sido confirmado.

A execução utiliza:

```python
await

In [ ]:
## Célula 26 — Código — execução principal
# ============================================================
# EXECUTAR PLANEJAMENTO
# ============================================================

async def executar_planejamento(
    scenario_id: str = PROJECT_NAME,
    primary_tech: str = "lte",
):
    request = {
        "scenario_id": scenario_id,
        "primary_tech": primary_tech,
    }

    resultado = await planner_orchestrator.execute(
        request,
        fetch_result=True,
        export=False,
    )

    return resultado


resultado_planejamento = await executar_planejamento()

print("=" * 80)
print("PLANEJAMENTO EXECUTADO")
print("=" * 80)
print()

print(json.dumps(
    resultado_planejamento,
    indent=2,
    ensure_ascii=False,
    default=str,
))

In [ ]:
# ============================================================
# VERIFICAR RESULTADOS ARMAZENADOS PELO ORCHESTRATOR
# ============================================================

print("=" * 80)
print("ESTADO DO PLANNER ORCHESTRATOR")
print("=" * 80)

print()

print("last_open_result:")
print(
    "OK"
    if planner_orchestrator.last_open_result is not None
    else "None"
)

print()

print("last_run_result:")
print(
    "OK"
    if planner_orchestrator.last_run_result is not None
    else "None"
)

print()

print("last_result:")
print(
    "OK"
    if planner_orchestrator.last_result is not None
    else "None"
)

print()

print("last_export_result:")
print(
    "OK"
    if planner_orchestrator.last_export_result is not None
    else "None"
)

# planning_result

O `planning_run` informa o andamento da execução.

A solução final, porém, deve ser obtida através de `planning_result`.

A estrutura esperada é:

```text
planning_result
├── status
├── scenario_id
└── result
    ├── planned_edges
    └── planned_nodes

In [ ]:
## Célula 29 — Código — mostrar `planning_result`
# ============================================================
# MOSTRAR planning_result FINAL
# ============================================================

planning_result = planner_orchestrator.last_result

if planning_result is None:
    raise RuntimeError(
        "planner_orchestrator.last_result está vazio. "
        "O planning_result não foi obtido."
    )

print(json.dumps(
    planning_result,
    indent=2,
    ensure_ascii=False,
    default=str,
))

In [ ]:
# ============================================================
# DIAGNÓSTICO DA SOLUÇÃO FINAL
# ============================================================

def diagnosticar_planning_result(planning_result):
    if not isinstance(planning_result, dict):
        raise TypeError(
            "planning_result precisa ser um dict."
        )

    payload = planning_result.get("result")

    if not isinstance(payload, dict):
        raise ValueError(
            "planning_result['result'] não é um dict."
        )

    planned_nodes = payload.get("planned_nodes", [])
    planned_edges = payload.get("planned_edges", [])

    if not isinstance(planned_nodes, list):
        raise ValueError(
            "planned_nodes precisa ser uma lista."
        )

    if not isinstance(planned_edges, list):
        raise ValueError(
            "planned_edges precisa ser uma lista."
        )

    print("=" * 80)
    print("DIAGNÓSTICO DA SOLUÇÃO")
    print("=" * 80)

    print()
    print(f"scenario_id     : {planning_result.get('scenario_id')}")
    print(f"status           : {planning_result.get('status')}")
    print(f"planned_nodes    : {len(planned_nodes)}")
    print(f"planned_edges    : {len(planned_edges)}")

    return {
        "planned_nodes": planned_nodes,
        "planned_edges": planned_edges,
    }


planning_solution = diagnosticar_planning_result(
    planning_result
)

# Preparação cartográfica

Os nós planejados retornados pelo planner utilizam:

```text
EPSG:31982

In [ ]:
## Célula 32 — Código — transformação e extração dos enlaces
# ============================================================
# EXTRAÇÃO CARTOGRÁFICA
# ============================================================

from pyproj import Transformer


PLANNING_CRS = "EPSG:31982"
MAP_CRS = "EPSG:4326"

planner_to_map = Transformer.from_crs(
    PLANNING_CRS,
    MAP_CRS,
    always_xy=True,
)


def extrair_dados_cartograficos(planning_result):
    if not isinstance(planning_result, dict):
        raise TypeError(
            "planning_result precisa ser um dict."
        )

    payload = planning_result.get("result")

    if not isinstance(payload, dict):
        raise ValueError(
            "planning_result['result'] não é um dict."
        )

    planned_nodes = payload.get("planned_nodes", [])
    planned_edges = payload.get("planned_edges", [])

    if not isinstance(planned_nodes, list):
        raise ValueError(
            "planned_nodes precisa ser uma lista."
        )

    if not isinstance(planned_edges, list):
        raise ValueError(
            "planned_edges precisa ser uma lista."
        )

    # --------------------------------------------------------
    # INDEXAR NÓS
    # --------------------------------------------------------

    node_index = {}

    for node in planned_nodes:

        if not isinstance(node, dict):
            continue

        node_id = node.get("node_id")
        x = node.get("x")
        y = node.get("y")

        if not node_id:
            continue

        if x is None or y is None:
            continue

        try:
            x = float(x)
            y = float(y)
        except (TypeError, ValueError):
            continue

        lon, lat = planner_to_map.transform(
            x,
            y,
        )

        node_index[str(node_id)] = {
            **node,
            "lat": float(lat),
            "lon": float(lon),
        }

    # --------------------------------------------------------
    # RESOLVER ENLACES
    # --------------------------------------------------------

    resolved_edges = []
    unresolved_edges = []

    for edge in planned_edges:

        if not isinstance(edge, dict):
            unresolved_edges.append(edge)
            continue

        src = edge.get("src")
        dst = edge.get("dst")

        src_node = node_index.get(str(src))
        dst_node = node_index.get(str(dst))

        if src_node is None or dst_node is None:
            unresolved_edges.append(edge)
            continue

        resolved_edges.append({
            **edge,
            "src_node": src_node,
            "dst_node": dst_node,
            "locations": [
                (
                    src_node["lat"],
                    src_node["lon"],
                ),
                (
                    dst_node["lat"],
                    dst_node["lon"],
                ),
            ],
        })

    return {
        "planned_nodes": list(node_index.values()),
        "planned_edges": planned_edges,
        "resolved_edges": resolved_edges,
        "unresolved_edges": unresolved_edges,
    }


map_data = extrair_dados_cartograficos(
    planning_result
)

print("=" * 80)
print("DADOS CARTOGRÁFICOS")
print("=" * 80)

print()
print(f"planned_nodes    : {len(map_data['planned_nodes'])}")
print(f"planned_edges    : {len(map_data['planned_edges'])}")
print(f"resolved_edges   : {len(map_data['resolved_edges'])}")
print(f"unresolved_edges : {len(map_data['unresolved_edges'])}")

In [ ]:
# ============================================================
# LISTAR ENLACES DA SOLUÇÃO FINAL
# ============================================================

print("=" * 80)
print("ENLACES PLANEJADOS")
print("=" * 80)

for index, edge in enumerate(
    map_data["resolved_edges"],
    start=1,
):
    print(
        f"{index:02d}. "
        f"{edge.get('src')} -> {edge.get('dst')} "
        f"| metric={edge.get('metric')}"
    )

# Sobreposição dos enlaces no mapa

Agora vamos adicionar os enlaces da solução final ao mapa existente.

Os marcadores dos `nodes.csv` permanecem no mapa.

Os enlaces serão adicionados como uma nova camada.

Cada enlace terá popup contendo:

- origem;
- destino;
- site de origem;
- site de destino;
- métrica do enlace.

In [ ]:
# ============================================================
# ADICIONAR ENLACES PLANEJADOS AO MAPA
# ============================================================

from ipyleaflet import LayerGroup, Polyline


def adicionar_planejamento_ao_mapa(
    mapa,
    map_data,
):
    resolved_edges = map_data.get(
        "resolved_edges",
        [],
    )

    layer = LayerGroup(
        name="Enlaces planejados",
    )

    for index, edge in enumerate(
        resolved_edges,
        start=1,
    ):
        src = edge.get("src")
        dst = edge.get("dst")
        metric = edge.get("metric")

        src_node = edge["src_node"]
        dst_node = edge["dst_node"]

        src_site = src_node.get("site_id")
        dst_site = dst_node.get("site_id")

        popup_html = f"""
        <b>Enlace planejado {index}</b><br>
        <br>
        <b>src:</b> {html.escape(str(src))}<br>
        <b>dst:</b> {html.escape(str(dst))}<br>
        <b>src_site:</b> {html.escape(str(src_site))}<br>
        <b>dst_site:</b> {html.escape(str(dst_site))}<br>
        <b>metric:</b> {html.escape(str(metric))}
        """

        popup = Popup(
            child=HTML(
                value=popup_html,
            ),
            close_button=True,
            auto_close=True,
        )

        line = Polyline(
            locations=edge["locations"],
            color="green",
            weight=4,
            opacity=0.9,
        )

        line.popup = popup

        layer.add(line)

    mapa.add(layer)

    return layer


planned_links_layer = adicionar_planejamento_ao_mapa(
    mapa_nodes,
    map_data,
)

print(
    f"{len(map_data['resolved_edges'])} "
    "enlaces adicionados ao mapa."
)

In [ ]:
# ============================================================
# MAPA FINAL
# ============================================================

mapa_nodes

# Execução completa em uma única chamada

A função abaixo reúne:

1. execução do planner;
2. obtenção do `planning_result`;
3. extração dos nós planejados;
4. resolução dos enlaces;
5. adição dos enlaces ao mapa.

Ela é útil para repetir o teste sem precisar executar manualmente todas as células intermediárias.

In [ ]:
# ============================================================
# EXECUTAR + PLOTAR
# ============================================================

async def executar_e_plotar_planejamento(
    scenario_id: str = PROJECT_NAME,
    primary_tech: str = "lte",
):
    # --------------------------------------------------------
    # 1. EXECUTAR PLANEJAMENTO
    # --------------------------------------------------------

    resultado = await executar_planejamento(
        scenario_id=scenario_id,
        primary_tech=primary_tech,
    )

    # --------------------------------------------------------
    # 2. OBTER RESULTADO FINAL
    # --------------------------------------------------------

    planning_result = planner_orchestrator.last_result

    if planning_result is None:
        raise RuntimeError(
            "O PlannerOrchestrator não armazenou "
            "o planning_result final."
        )

    # --------------------------------------------------------
    # 3. EXTRAIR DADOS CARTOGRÁFICOS
    # --------------------------------------------------------

    map_data = extrair_dados_cartograficos(
        planning_result
    )

    # --------------------------------------------------------
    # 4. ADICIONAR AO MAPA
    # --------------------------------------------------------

    adicionar_planejamento_ao_mapa(
        mapa_nodes,
        map_data,
    )

    # --------------------------------------------------------
    # 5. RESUMO
    # --------------------------------------------------------

    print("=" * 80)
    print("PLANEJAMENTO + MAPA")
    print("=" * 80)

    print()
    print(
        f"scenario_id     : "
        f"{planning_result.get('scenario_id')}"
    )

    print(
        f"status           : "
        f"{planning_result.get('status')}"
    )

    print(
        f"planned_nodes    : "
        f"{len(map_data['planned_nodes'])}"
    )

    print(
        f"planned_edges    : "
        f"{len(map_data['planned_edges'])}"
    )

    print(
        f"resolved_edges   : "
        f"{len(map_data['resolved_edges'])}"
    )

    print(
        f"unresolved_edges : "
        f"{len(map_data['unresolved_edges'])}"
    )

    return {
        "execution_result": resultado,
        "planning_result": planning_result,
        "map_data": map_data,
    }

# Executar novamente o planejamento

Esta é a célula que efetivamente dispara uma nova execução.

Ela é propositalmente manual.

Não deve ser executada automaticamente quando o notebook é aberto.

In [ ]:
# ============================================================
# EXECUÇÃO FINAL
# ============================================================

resultado_mapa = await executar_e_plotar_planejamento()

In [ ]:
mapa_nodes

In [ ]:
# ============================================================
# RESUMO FINAL
# ============================================================

planning_result = resultado_mapa["planning_result"]
map_data = resultado_mapa["map_data"]

print("=" * 80)
print("RESUMO FINAL DO PLANEJAMENTO")
print("=" * 80)

print()
print(f"Projeto          : {PROJECT_NAME}")
print(f"Cenário          : {planning_result.get('scenario_id')}")
print(f"Status           : {planning_result.get('status')}")

print()
print("Solução final:")
print(
    f"  planned_nodes  : "
    f"{len(map_data['planned_nodes'])}"
)
print(
    f"  planned_edges  : "
    f"{len(map_data['planned_edges'])}"
)
print(
    f"  resolved_edges : "
    f"{len(map_data['resolved_edges'])}"
)
print(
    f"  unresolved     : "
    f"{len(map_data['unresolved_edges'])}"
)

print()
print(f"scenario.toml:")
print(f"  {SCENARIO_PATH}")

print()
print(f"nodes.csv:")
print(f"  {NODES_PATH}")

# Exportação opcional

O planejamento já foi executado e o `planning_result` já está disponível.

A exportação não é necessária para visualizar os enlaces no mapa.

Se quisermos testar posteriormente o `planning_export`, podemos fazê-lo separadamente através do `PlannerOrchestrator`, sem alterar o fluxo cartográfico.

In [ ]:
# ============================================================
# EXPORTAÇÃO OPCIONAL
# ============================================================

async def exportar_planejamento():
    resultado = await planner_orchestrator.planning_export(
        {
            "scenario_id": PROJECT_NAME,
            "planner": "cell",
            "rank_threshold": 10,
            "primary_tech": "lte",
            "solution_kind": "graph",
        }
    )

    print("=" * 80)
    print("PLANNING EXPORT")
    print("=" * 80)
    print()

    print(json.dumps(
        resultado,
        indent=2,
        ensure_ascii=False,
        default=str,
    ))

    return resultado


# Execute somente quando quiser testar a exportação:
#
# resultado_export = await exportar_planejamento()

# Encerramento / limpeza

O `PlannerOrchestrator` pode ter criado sua própria conexão MCP.

O fechamento deve acontecer no mesmo contexto assíncrono em que a conexão foi utilizada.

Não devemos fechar o MCP dentro de uma thread ou callback diferente.

Por isso a limpeza fica explícita e manual no final do notebook.

In [ ]:
# ============================================================
# CLEANUP DO PLANNER
# ============================================================

async def fechar_planner():
    await planner_orchestrator.close()
    print("PlannerOrchestrator encerrado.")


# Execute somente quando terminar os testes:
#
# await fechar_planner()